In [1]:
!pip install ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def extract_faces_from_social_media():
    SOCIAL_DIR = "/content/drive/MyDrive/Final_Project/dataset/enrollment/social_media"

    for rollno in os.listdir(SOCIAL_DIR):
        roll_dir = os.path.join(SOCIAL_DIR, rollno)
        if not os.path.isdir(roll_dir):
            continue

        out_dir = os.path.join(OUTPUT_DIR, rollno)
        ensure_dir(out_dir)

        existing_count = len(os.listdir(out_dir))

        for img_name in os.listdir(roll_dir):
            if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                continue

            img_path = os.path.join(roll_dir, img_name)
            img = cv2.imread(img_path)

            if img is None:
                continue

            results = model(img, verbose=False)[0]

            # Accept only images with exactly one face
            if len(results.boxes) != 1:
                continue

            box = results.boxes[0]
            conf = float(box.conf[0])
            if conf < CONF_THRESH:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            face = img[y1:y2, x1:x2]

            if face.size == 0:
                continue

            h, w = face.shape[:2]
            if h < MIN_FACE_SIZE or w < MIN_FACE_SIZE:
                continue

            filename = f"{rollno}_social_{existing_count:05d}.jpg"
            cv2.imwrite(os.path.join(out_dir, filename), face)
            existing_count += 1

        print(f"[INFO] Social media faces extracted for {rollno}")


In [4]:
import cv2
import os
from ultralytics import YOLO

# ---------------- CONFIG ---------------- #
VIDEO_DIR = "/content/drive/MyDrive/Final_Project/dataset/enrollment/phone_videos"
OUTPUT_DIR = "/content/drive/MyDrive/Final_Project/dataset/enrollment/faces_raw"
MODEL_PATH = "/content/drive/MyDrive/Final_Project/models/yolov8n-face.pt"

# Optimized settings for maximum photos (40-50+)
EXTRACT_INTERVAL_SECONDS = 0.25  # Extract every 0.25 seconds (4 per second)
CONF_THRESH = 0.65 # Lower threshold to detect more faces
MIN_FACE_SIZE = 70  # Accept smaller faces
TARGET_FACE_SIZE = (224, 224)  # Resize all faces to consistent size
# ---------------------------------------- #

model = YOLO(MODEL_PATH)

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def extract_faces(video_path, rollno):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    print(f"  Video info: {duration:.1f}s duration, {fps:.1f} FPS, {total_frames} frames")

    # Calculate frame interval based on time
    frame_interval = int(fps * EXTRACT_INTERVAL_SECONDS)
    frame_interval = max(1, frame_interval)  # At least 1 frame

    out_dir = os.path.join(OUTPUT_DIR, rollno)
    ensure_dir(out_dir)

    frame_count = 0
    saved_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Process frames at time-based intervals
        if frame_count % frame_interval == 0:
            results = model(frame, verbose=False)[0]

            # Find the largest face (assume it's the enrolled person)
            best_face = None
            best_area = 0

            for box in results.boxes:
                conf = float(box.conf[0])
                if conf < CONF_THRESH:
                    continue

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                w, h = x2 - x1, y2 - y1

                # Check minimum size
                if h < MIN_FACE_SIZE or w < MIN_FACE_SIZE:
                    continue

                # Select largest face (closest to camera)
                area = w * h
                if area > best_area:
                    best_area = area
                    best_face = (x1, y1, x2, y2)

            # Save the best face from this frame
            if best_face is not None:
                x1, y1, x2, y2 = best_face
                face = frame[y1:y2, x1:x2]

                if face.size > 0:
                    # Resize to consistent dimensions
                    face_resized = cv2.resize(face, TARGET_FACE_SIZE)

                    filename = f"{rollno}_{saved_count:03d}.jpg"
                    cv2.imwrite(os.path.join(out_dir, filename), face_resized)
                    saved_count += 1

        frame_count += 1

    cap.release()
    print(f"[INFO] Extracted {saved_count} faces for {rollno}")

    # Warning if too few faces extracted
    if saved_count < 30:
        print(f"[WARNING] Only {saved_count} faces extracted! Consider:")
        print(f"  - Lowering CONF_THRESH (current: {CONF_THRESH})")
        print(f"  - Lowering MIN_FACE_SIZE (current: {MIN_FACE_SIZE})")
        print(f"  - Decreasing EXTRACT_INTERVAL_SECONDS (current: {EXTRACT_INTERVAL_SECONDS})")

def main():
    video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")]

    if not video_files:
        print(f"[ERROR] No .mp4 files found in {VIDEO_DIR}")
        return

    print(f"[INFO] Found {len(video_files)} videos to process")
    print(f"[INFO] Target: 40-50+ faces per video")
    print(f"[INFO] Extraction interval: {EXTRACT_INTERVAL_SECONDS}s (4 frames per second)")

    for video_file in video_files:
        rollno = os.path.splitext(video_file)[0]
        video_path = os.path.join(VIDEO_DIR, video_file)

        print(f"\n[PROCESSING] {rollno}")
        extract_faces(video_path, rollno)

if __name__ == "__main__":
    main()

[INFO] Found 1 videos to process
[INFO] Target: 40-50+ faces per video
[INFO] Extraction interval: 0.25s (4 frames per second)

[PROCESSING] 780314
  Video info: 112.7s duration, 30.0 FPS, 3382 frames
[INFO] Extracted 396 faces for 780314


yesle chai dlib ko alignment garne wala download gareko

In [5]:
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2
!mv shape_predictor_68_face_landmarks.dat /content/drive/MyDrive/Final_Project/models/

--2026-02-13 14:47:40--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-02-13 14:47:40--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  45.4MB/s    in 1.3s    

2026-02-13 14:47:42 (45.4 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]



yesle chai dlib use gareko xa alignment ko lagi

In [9]:
import cv2
import os
import numpy as np
from ultralytics import YOLO
import dlib

# ---------------- CONFIG ----------------
VIDEO_DIR = "/content/drive/MyDrive/Final_Project/dataset/enrollment/phone_videos"
OUTPUT_DIR = "/content/drive/MyDrive/Final_Project/dataset/enrollment/faces_raw1"
MODEL_PATH = "/content/drive/MyDrive/Final_Project/models/yolov8n-face.pt"

# Optimized settings for maximum photos (40-50+)
EXTRACT_INTERVAL_SECONDS = 0.25  # Extract every 0.25 seconds (4 per second)
CONF_THRESH = 0.65  # Lower threshold to detect more faces
MIN_FACE_SIZE = 70  # Accept smaller faces
TARGET_FACE_SIZE = (224, 224)  # Resize all faces to consistent size
FACE_PADDING = 0.10  # 10% padding around face

# Alignment settings
MAX_ROTATION = 30  # Maximum head tilt angle in degrees (reject if more tilted)
MIN_EYE_DISTANCE = 25  # Minimum pixel distance between eyes
MIN_ASPECT_RATIO = 0.7  # Minimum width/height ratio (reject partial/cut-off faces)
MAX_ASPECT_RATIO = 1.3  # Maximum width/height ratio

# ----------------------------------------

model = YOLO(MODEL_PATH)

# Load dlib's face landmark detector for alignment
try:
    predictor_path = "/content/drive/MyDrive/Final_Project/models/shape_predictor_68_face_landmarks.dat"
    predictor = dlib.shape_predictor(predictor_path)
    use_alignment = True
    print("[INFO] Face alignment enabled")
except:
    use_alignment = False
    print("[WARNING] Landmark detector not found. Alignment disabled.")
    print("Download from: http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2")


def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)


def get_face_landmarks(image, face_rect):
    """Get 68 facial landmarks using dlib for a specific face region"""
    if not use_alignment:
        return None

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Convert face_rect (x1, y1, x2, y2) to dlib rectangle
    x1, y1, x2, y2 = face_rect
    rect = dlib.rectangle(int(x1), int(y1), int(x2), int(y2))

    try:
        landmarks = predictor(gray, rect)
        points = []
        for i in range(68):
            points.append((landmarks.part(i).x, landmarks.part(i).y))
        return np.array(points)
    except:
        return None


def check_face_alignment(landmarks):
    """Check if face is properly aligned based on eye positions"""
    if landmarks is None:
        return True, 0  # Skip check if no landmarks

    # Get eye coordinates (landmarks 36-41: left eye, 42-47: right eye)
    left_eye = landmarks[36:42].mean(axis=0)
    right_eye = landmarks[42:48].mean(axis=0)

    # Calculate eye distance
    eye_distance = np.linalg.norm(left_eye - right_eye)
    if eye_distance < MIN_EYE_DISTANCE:
        return False, f"eyes_too_close ({eye_distance:.1f})"

    # Calculate rotation angle
    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    if abs(angle) > MAX_ROTATION:
        return False, f"tilted ({abs(angle):.1f}°)"

    return True, abs(angle)


def is_full_face(face_box, frame_shape):
    """Check if face is not cut off at frame edges"""
    x1, y1, x2, y2 = face_box
    frame_h, frame_w = frame_shape[:2]

    # Check if face touches frame edges (indicating it might be cut off)
    margin = 5  # 5 pixel margin from edge
    if x1 <= margin or y1 <= margin or x2 >= (frame_w - margin) or y2 >= (frame_h - margin):
        return False, "edge_cutoff"

    # Check aspect ratio (full frontal faces should be roughly square)
    w = x2 - x1
    h = y2 - y1
    aspect_ratio = w / h if h > 0 else 0

    if aspect_ratio < MIN_ASPECT_RATIO or aspect_ratio > MAX_ASPECT_RATIO:
        return False, f"bad_aspect ({aspect_ratio:.2f})"

    return True, "good"


def align_and_crop_face(frame, face_box, landmarks):
    """
    Align the entire frame based on eye positions, then crop the face.
    This prevents the face from being cut off during rotation.
    """
    x1, y1, x2, y2 = face_box

    # If no landmarks, just crop without alignment
    if landmarks is None or not use_alignment:
        # Add padding
        w, h = x2 - x1, y2 - y1
        padding_w = int(w * FACE_PADDING)
        padding_h = int(h * FACE_PADDING)

        x1 = max(0, x1 - padding_w)
        y1 = max(0, y1 - padding_h)
        x2 = min(frame.shape[1], x2 + padding_w)
        y2 = min(frame.shape[0], y2 + padding_h)

        face = frame[y1:y2, x1:x2]
        return face

    # Get eye centers for alignment
    left_eye = landmarks[36:42].mean(axis=0)
    right_eye = landmarks[42:48].mean(axis=0)

    # Calculate rotation angle
    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    # Get rotation matrix centered between eyes
    eye_center = ((left_eye[0] + right_eye[0]) / 2, (left_eye[1] + right_eye[1]) / 2)

    # Rotate the entire frame
    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    aligned_frame = cv2.warpAffine(frame, M, (frame.shape[1], frame.shape[0]))

    # Transform face bounding box coordinates
    # Create corners of the bounding box
    corners = np.array([
        [x1, y1, 1],
        [x2, y1, 1],
        [x2, y2, 1],
        [x1, y2, 1]
    ], dtype=np.float32)

    # Apply rotation to corners
    transformed_corners = M.dot(corners.T).T

    # Get new bounding box from transformed corners
    new_x1 = int(np.min(transformed_corners[:, 0]))
    new_y1 = int(np.min(transformed_corners[:, 1]))
    new_x2 = int(np.max(transformed_corners[:, 0]))
    new_y2 = int(np.max(transformed_corners[:, 1]))

    # Add padding
    w, h = new_x2 - new_x1, new_y2 - new_y1
    padding_w = int(w * FACE_PADDING)
    padding_h = int(h * FACE_PADDING)

    new_x1 = max(0, new_x1 - padding_w)
    new_y1 = max(0, new_y1 - padding_h)
    new_x2 = min(aligned_frame.shape[1], new_x2 + padding_w)
    new_y2 = min(aligned_frame.shape[0], new_y2 + padding_h)

    # Crop the aligned face
    face = aligned_frame[new_y1:new_y2, new_x1:new_x2]

    return face


def extract_faces(video_path, rollno):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    print(f"  Video info: {duration:.1f}s duration, {fps:.1f} FPS, {total_frames} frames")

    # Calculate frame interval based on time
    frame_interval = int(fps * EXTRACT_INTERVAL_SECONDS)
    frame_interval = max(1, frame_interval)  # At least 1 frame

    out_dir = os.path.join(OUTPUT_DIR, rollno)
    ensure_dir(out_dir)

    frame_count = 0
    saved_count = 0
    rejected = {
        "tilted": 0,
        "eyes_too_close": 0,
        "no_landmarks": 0,
        "low_conf": 0,
        "too_small": 0,
        "edge_cutoff": 0,
        "bad_aspect": 0
    }

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Process frames at time-based intervals
        if frame_count % frame_interval == 0:
            results = model(frame, verbose=False)[0]

            # Find the largest face (assume it's the enrolled person)
            best_face = None
            best_area = 0
            best_box = None

            for box in results.boxes:
                conf = float(box.conf[0])
                if conf < CONF_THRESH:
                    rejected["low_conf"] += 1
                    continue

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                w, h = x2 - x1, y2 - y1

                # Check minimum size
                if h < MIN_FACE_SIZE or w < MIN_FACE_SIZE:
                    rejected["too_small"] += 1
                    continue

                # Check if it's a full face (not cut off)
                is_full, reason = is_full_face((x1, y1, x2, y2), frame.shape)
                if not is_full:
                    reject_type = reason.split('(')[0].strip()
                    if reject_type in rejected:
                        rejected[reject_type] += 1
                    continue

                # Select largest face (closest to camera)
                area = w * h
                if area > best_area:
                    best_area = area
                    best_box = (x1, y1, x2, y2)

            # Process the best face from this frame
            if best_box is not None:
                x1, y1, x2, y2 = best_box

                # Get landmarks on the ORIGINAL frame (before alignment)
                landmarks = get_face_landmarks(frame, best_box)

                if use_alignment:
                    if landmarks is None:
                        rejected["no_landmarks"] += 1
                        frame_count += 1
                        continue

                    # Check alignment quality
                    is_aligned, info = check_face_alignment(landmarks)
                    if not is_aligned:
                        # Track rejection reason
                        reject_type = info.split('(')[0].strip()
                        if reject_type in rejected:
                            rejected[reject_type] += 1
                        frame_count += 1
                        continue

                # NOW: Align the entire frame first, then crop
                face = align_and_crop_face(frame, best_box, landmarks)

                if face.size > 0:
                    # Resize to consistent dimensions
                    face_resized = cv2.resize(face, TARGET_FACE_SIZE)

                    filename = f"{rollno}_{saved_count:03d}.jpg"
                    cv2.imwrite(os.path.join(out_dir, filename), face_resized)
                    saved_count += 1

        frame_count += 1

    cap.release()

    print(f"[INFO] Extracted {saved_count} faces for {rollno}")
    if use_alignment:
        print(f"  Rejected - Tilted: {rejected['tilted']}, Eyes close: {rejected['eyes_too_close']}, "
              f"Cutoff: {rejected['edge_cutoff']}, Bad aspect: {rejected['bad_aspect']}, "
              f"No landmarks: {rejected['no_landmarks']}, Low conf: {rejected['low_conf']}, "
              f"Too small: {rejected['too_small']}")

    # Warning if too few faces extracted
    if saved_count < 30:
        print(f"[WARNING] Only {saved_count} faces extracted! Consider:")
        print(f"  - Lowering CONF_THRESH (current: {CONF_THRESH})")
        print(f"  - Lowering MIN_FACE_SIZE (current: {MIN_FACE_SIZE})")
        print(f"  - Increasing MAX_ROTATION (current: {MAX_ROTATION})")
        print(f"  - Relaxing aspect ratio (current: {MIN_ASPECT_RATIO}-{MAX_ASPECT_RATIO})")
        print(f"  - Decreasing EXTRACT_INTERVAL_SECONDS (current: {EXTRACT_INTERVAL_SECONDS})")


def main():
    video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")]

    if not video_files:
        print(f"[ERROR] No .mp4 files found in {VIDEO_DIR}")
        return

    print(f"[INFO] Found {len(video_files)} videos to process")
    print(f"[INFO] Target: 40-50+ faces per video")
    print(f"[INFO] Extraction interval: {EXTRACT_INTERVAL_SECONDS}s (4 frames per second)")
    print(f"[INFO] Face padding: {int(FACE_PADDING * 100)}%")
    print(f"[INFO] Alignment: {'Enabled' if use_alignment else 'Disabled'}")
    if use_alignment:
        print(f"[INFO] Max rotation: {MAX_ROTATION}°, Min eye distance: {MIN_EYE_DISTANCE}px")
        print(f"[INFO] Aspect ratio: {MIN_ASPECT_RATIO}-{MAX_ASPECT_RATIO}\n")

    for video_file in video_files:
        rollno = os.path.splitext(video_file)[0]
        video_path = os.path.join(VIDEO_DIR, video_file)

        print(f"\n[PROCESSING] {rollno}")
        extract_faces(video_path, rollno)


if __name__ == "__main__":
    main()

[INFO] Face alignment enabled
[INFO] Found 14 videos to process
[INFO] Target: 40-50+ faces per video
[INFO] Extraction interval: 0.25s (4 frames per second)
[INFO] Face padding: 10%
[INFO] Alignment: Enabled
[INFO] Max rotation: 30°, Min eye distance: 25px
[INFO] Aspect ratio: 0.7-1.3


[PROCESSING] 780322
  Video info: 14.7s duration, 29.9 FPS, 438 frames
[INFO] Extracted 5 faces for 780322
  Rejected - Tilted: 0, Eyes close: 0, Cutoff: 0, Bad aspect: 0, No landmarks: 0, Low conf: 51, Too small: 0
[WARNING] Only 5 faces extracted! Consider:
  - Lowering CONF_THRESH (current: 0.65)
  - Lowering MIN_FACE_SIZE (current: 70)
  - Increasing MAX_ROTATION (current: 30)
  - Relaxing aspect ratio (current: 0.7-1.3)
  - Decreasing EXTRACT_INTERVAL_SECONDS (current: 0.25)

[PROCESSING] 780306
  Video info: 12.0s duration, 29.9 FPS, 358 frames
[INFO] Extracted 8 faces for 780306
  Rejected - Tilted: 0, Eyes close: 0, Cutoff: 0, Bad aspect: 32, No landmarks: 0, Low conf: 13, Too small: 0
[WARNING